# MAE pretraining

Пайплайн предобучения MAE-энкодера на термо-кадрах.

In [7]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

print("cwd:", PROJECT_ROOT)

cwd: c:\Users\Iudin\Documents\Visual Studio Projects\Projects\thermal-control-ya-project


In [8]:
import torch

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger
from torch.utils.data import DataLoader

from datasets import TermoFrameDataset
from datasets.transforms import Compose, HorizontalFlip, VerticalFlip, RandomRotate90

from models.pretraining.lightning_module import MAELightningModule

## DataLoader'ы

`train_datasets`/`val_datasets` — разные поддатасеты/видео, чтобы val не совпадал с train.

In [9]:
transform = Compose([
    HorizontalFlip(p=0.5),
    VerticalFlip(p=0.5),
    RandomRotate90(p=0.5),
])

train_ds = TermoFrameDataset(root_dir=str(PROJECT_ROOT / "datasets" / "datasets_list"), include="dataset_kaggle", standard_size=(128, 128), transform=transform)
val_ds = TermoFrameDataset(root_dir=str(PROJECT_ROOT / "datasets" / "datasets_list"), include="dataset_tpu", standard_size=(128, 128), transform=None)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=1, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=1, pin_memory=True)

In [10]:
len(train_ds)

1900

## Модель

In [11]:
model = MAELightningModule(
    img_size=128, patch_size=16, in_channels=1,
    embed_dim=64, depth=2, num_heads=4,
    decoder_dim=32, decoder_depth=1, decoder_heads=2,
    mask_ratio=0.4, lr=1.5e-4,
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_last=True,
    filename="epoch{epoch:02d}-loss{val_loss:.4f}",
)
logger = CSVLogger("logs", name="mae_pretrain")

trainer = pl.Trainer(
    max_epochs=10, accelerator="cpu",
    devices=1, callbacks=[checkpoint_callback],
    logger=logger,
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [12]:
trainer.fit(model, train_loader, val_loader)


  | Name      | Type                  | Params | Mode  | FLOPs
--------------------------------------------------------------------
0 | model     | MAE                   | 139 K  | train | 0    
1 | criterion | MAEReconstructionLoss | 0      | train | 0    
--------------------------------------------------------------------
139 K     Trainable params
0         Non-trainable params
139 K     Total params
0.559     Total estimated model params size (MB)
39        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

c:\Users\Iudin\Documents\Visual Studio Projects\Projects\thermal-control-ya-project\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3783: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Перенос энкодера в downstream-задачу

In [ ]:
pretrained_encoder_state = model.model.encoder.state_dict()
torch.save(pretrained_encoder_state, "checkpoints/mae_encoder.pt")